<div style="text-align: justify;">

# Notebook 4: Arquitectura del Sistema. Exploración de Módulos (`src` y `app`)

Este notebook tiene como propósito ofrecer una visión general y estructurada de la arquitectura del proyecto. A través de este documento, exploraremos visualmente la jerarquía de archivos y carpetas que componen el núcleo de la aplicación, dividiendo nuestro análisis en los dos directorios principales: `src` (que contiene el motor analítico) y `app` (que gestiona la API, la base de datos y los servicios). El objetivo no es profundizar en el código línea por línea, sino proporcionar un mapa claro de dónde reside cada responsabilidad dentro del sistema.

## Índice

1. [Preparación del Entorno y Herramientas](#preparacion)
2. [Estructura y Componentes del Directorio `src`](#src)
3. [Estructura y Componentes del Directorio `app`](#app)

</div>

<div style="text-align: justify;">

## 1. Preparación del Entorno y Herramientas <a id="preparacion"></a>

En esta celda se configura la ejecución desde el directorio raíz del proyecto, dado que este documento se encuentra en una carpeta dentro de este directorio raíz, hemos de retroceder un nivel. Se define además la función que servirá para ilustrar la organización de los directorios objetivo del análisis.

</div>

In [1]:
import os
import sys

# 1. Configuración del Pathing
# Al ejecutarse desde la carpeta 'notebooks', retrocedemos un nivel para situarnos en la raíz del proyecto
notebook_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(notebook_dir, ".."))

if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

os.chdir(root_dir)

# 2. Función generadora de árboles de directorio
def generar_arbol_proyecto(directorio_raiz, carpeta_objetivo):
    """
    Genera una representación visual en forma de árbol de los archivos .py 
    dentro de una carpeta específica, omitiendo cachés.
    """
    ruta_carpeta = os.path.join(directorio_raiz, carpeta_objetivo)
    if not os.path.exists(ruta_carpeta):
        print(f"La carpeta '{carpeta_objetivo}' no existe en la ruta especificada.")
        return
        
    print(f"📂 {carpeta_objetivo}/")
    
    for raiz, directorios, archivos in os.walk(ruta_carpeta):
        # Omitir carpetas internas de caché (__pycache__)
        directorios[:] = [d for d in directorios if not d.startswith("__")]
        archivos_py = sorted([f for f in archivos if f.endswith('.py')])
        
        if not archivos_py and raiz != ruta_carpeta:
            continue
            
        # Calcular nivel de indentación relativo
        nivel = raiz.replace(ruta_carpeta, '').count(os.sep)
        indentacion = '    ' * nivel
        
        subcarpeta = os.path.basename(raiz)
        if subcarpeta != carpeta_objetivo:
            print(f"{indentacion}├── 📁 {subcarpeta}/")
            indentacion_archivo = '    ' * (nivel + 1)
        else:
            indentacion_archivo = '    '
            
        for idx, archivo in enumerate(archivos_py):
            es_ultimo = (idx == len(archivos_py) - 1)
            conector = "└── 📄" if es_ultimo else "├── 📄"
            print(f"{indentacion_archivo}{conector} {archivo}")

<div style="text-align: justify;">

## 2. Estructura y Componentes del Directorio `src` <a id="src"></a>

En la siguiente celda generamos, a partir de la función definida en la sección de [preparación del entorno](#preparacion), el árbol de la carpeta `src`.

</div>

In [2]:
generar_arbol_proyecto(".", "src")

📂 src/
    ├── 📁 core/
        ├── 📄 __init__.py
        ├── 📄 context.py
        ├── 📄 orchestrator.py
        ├── 📄 registry.py
        └── 📄 sandbox_injector.py
    ├── 📁 layer1_ingestion/
        ├── 📄 __init__.py
        ├── 📄 aligner.py
        ├── 📄 assembler.py
        └── 📄 data_ingestor.py
    ├── 📁 layer2_routing/
        ├── 📄 __init__.py
        ├── 📄 evaluator.py
        ├── 📄 route_types.py
        └── 📄 router.py
    ├── 📁 layer3_models/
        ├── 📄 __init__.py
        ├── 📄 detector.py
        ├── 📄 external_wrapper.py
        ├── 📄 feature_engineering.py
        ├── 📄 feature_selection.py
        ├── 📄 forecast_manager.py
        ├── 📄 forecaster.py
        ├── 📄 foundation_manager.py
        ├── 📄 imputer.py
        ├── 📄 semantic_validator.py
        └── 📄 thresholder.py
    ├── 📁 layer4_filtering/
        ├── 📄 __init__.py
        └── 📄 filter.py
    ├── 📁 layer5_mlops/
        ├── 📄 __init__.py
        ├── 📄 architectural_monitor.py
        ├── 📄 drift_analyzer.

<div style="text-align: justify;">

### Descripción de los módulos en `src`

La carpeta `src` contiene el motor analítico del proyecto. Está estructurada en capas que representan el flujo de datos desde la ingesta hasta la emisión del diagnóstico:

* **`core/`**: Contiene los cimientos del sistema. Incluye el orquestador de las ejecuciones (`orchestrator.py`), la definición del contexto de datos que viaja entre capas y las preferencias del usuario (`context.py`), un simulador de bases de datos, utilizado para pruebas o como fallback en caso de que no se disponga de conexión con la base de datos real (`registry.py`), y el inyector de daños sintéticos para las pruebas aisladas en el entorno Sandbox (`sandbox_injector.py`).
* **`layer1_ingestion/`**: Se encarga de la entrada y preparación inicial de las series temporales. Abarca la ingesta de datos (`data_ingestor.py`), el alineamiento temporal (`aligner.py`) y el ensamblado de las señales en un único dataset de múltiples señales (`assembler.py`).
* **`layer2_routing/`**: Decide de forma dinámica qué ruta debe tomar una señal basándose en sus características. Contiene el evaluador (`evaluator.py`) y el enrutador principal (`router.py`), apoyado sobre las definiciones de las rutas y sus características, definidas en `route_types.py`.
* **`layer3_models/`**: Es el núcleo algorítmico. Aquí residen las clases que manejan la detección de anomalías (`detector.py`), la imputación de huecos (`imputer.py`), la generación de predicciones futuras (`forecaster.py` y `forecast_manager.py`), el manejo de las arquitecturas fundacionales de Deep LEarning utilizadas tanto como imputadores como detectores (`foundation_manager.py`), y la ingeniería/selección de características (`feature_engineering.py`, `feature_selection.py`). También incluye validadores semánticos y thresholders para el cálculo de umbrales. El `external_wrapper.py` se encarga de la gestión de modelos externos, que se puedes cargar vía preferencias con sus parámetros definidos, vía API, o vía archivo `.onnx`.
* **`layer4_filtering/`**: Contiene el filtrado de ruido estocástico mediante el filtro de Savitzky-Golay (`filter.py`).
* **`layer5_mlops/`**: Gestiona el ciclo de vida continuo de los modelos en producción. Incluye el motor de optimización de hiperparámetros (`hpo_engine.py`), el análisis de deriva de datos (`drift_analyzer.py`) y el orquestador de reentrenamientos automáticos (`retraining_orchestrator.py`), estos dos últimos dirigidos por el monitor arquitectónico (`architectural_monitor.py`).

</div>

<div style="text-align: justify;">

## 3. Estructura y Componentes del Directorio `app` <a id="app"></a>

En la siguiente celda generamos, a partir de la función definida en la sección de [preparación del entorno](#preparacion), el árbol de la carpeta `app`.

</div>

In [3]:
generar_arbol_proyecto(".", "app")

📂 app/
    ├── 📄 __init__.py
    ├── 📄 config.py
    ├── 📄 database.py
    └── 📄 main.py
    ├── 📁 api/
        └── 📄 __init__.py
        ├── 📁 v1/
            ├── 📄 __init__.py
            └── 📄 router.py
            ├── 📁 endpoints/
                ├── 📄 __init__.py
                ├── 📄 datasets.py
                ├── 📄 jobs.py
                ├── 📄 metrics.py
                ├── 📄 mlops.py
                ├── 📄 models.py
                └── 📄 sources.py
    ├── 📁 core/
        ├── 📄 __init__.py
        ├── 📄 exceptions.py
        ├── 📄 logging.py
        └── 📄 middleware.py
        ├── 📁 storage/
            ├── 📄 __init__.py
            ├── 📄 base.py
            ├── 📄 local.py
            ├── 📄 manager.py
            └── 📄 s3.py
    ├── 📁 models/
        ├── 📄 __init__.py
        └── 📄 orm.py
    ├── 📁 schemas/
        ├── 📄 __init__.py
        ├── 📄 dataset.py
        ├── 📄 datetime_types.py
        ├── 📄 enums.py
        ├── 📄 job.py
        ├── 📄 metrics.py
        ├── 📄 model.

<div style="text-align: justify;">

### Descripción de los módulos en `app`

La carpeta `app` representa la capa de infraestructura y exposición hacia el exterior (API). Es la encargada de recibir las peticiones de los usuarios, interactuar con la base de datos y lanzar los procesos definidos en `src`:

* **Raíz de `app` (`main.py`, `config.py`, `database.py`)**: Aquí se inicializa la aplicación FastAPI (`main.py`), se definen las variables de entorno y configuración general (`config.py`) y se establece la conexión con el motor de base de datos (`database.py`).
* **`api/v1/endpoints/`**: Contiene los controladores que exponen las rutas de la API. Se divide por dominios como `jobs.py` (tareas asíncronas), `datasets.py` (manejo de conjuntos de datos), `models.py` (gestión del catálogo de modelos) y `mlops.py` (operaciones de mantenimiento).
* **`core/`**: Aloja configuraciones de la API como el manejo de excepciones (`exceptions.py`), configuración de logs (`logging.py`), middlewares de red (`middleware.py`) y el almacenamiento de archivos en distintos entornos (locales, en la nube...) (`storage/`).
* **`models/`**: Define los esquemas y entidades de la base de datos a través de Object-Relational Mapping (ORM) mediante SQLAlchemy (`orm.py`).
* **`schemas/`**: Alberga los modelos de validación basados en Pydantic. Aquí se define la estructura de las peticiones y respuestas de la API, incluyendo enumeradores (`enums.py`), métricas (`metrics.py`) y definiciones de trabajos (`job.py`).
* **`services/`**: Actúa como la capa de unión entre la API (`endpoints`) y el núcleo (`src`). Destacan `job_service.py` (orquesta auditorías y optimizaciones en segundo plano), `training_service.py` (gestiona los flujos de entrenamiento) y los servicios de MLOps (`mlops_service.py`, `mlops_scheduler.py`).

</div>